In [6]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("aslak")
from grinsted_firn_model import *


def cubic_root_ultrafast(k3, k2, k1, k0):
    # might not be stable
    a = k2 / k3
    b = k1 / k3
    c = k0 / k3

    a3 = a / 3.0
    p = b - a * a3
    q = 2.0 * a3**3 - a3 * b + c

    D = (0.5 * q) ** 2 + (p / 3.0) ** 3
    sqrt_D = np.sqrt(D)

    u = np.cbrt(-0.5 * q + sqrt_D)
    v = np.cbrt(-0.5 * q - sqrt_D)

    return (u + v) - a3


@numba.njit(inline="always", fastmath=True)
def cubic_root_closest_real_to_zero(k3, k2, k1, k0):
    # -----------------------------
    # Scale for numerical stability
    scale = max(abs(k0), abs(k1), abs(k2), abs(k3))
    if scale > 0.0:
        k3 /= scale
        k2 /= scale
        k1 /= scale
        k0 /= scale
    # -----------------------------

    # --- Degenerate cases ---
    if abs(k3) < 1e-50:
        if abs(k2) < 1e-50:
            if abs(k1) < 1e-50:
                return 0.0
            return -k0 / k1  # linear

        disc = k1 * k1 - 4.0 * k2 * k0
        if disc < 0.0:
            disc = 0.0

        r1 = (-k1 + np.sqrt(disc)) / (2.0 * k2)
        r2 = (-k1 - np.sqrt(disc)) / (2.0 * k2)

        return r1 if abs(r1) < abs(r2) else r2

    # --- Normalize cubic ---
    inv_k3 = 1.0 / k3
    b = k2 * inv_k3
    c = k1 * inv_k3
    d = k0 * inv_k3

    # depressed cubic: x = y - b/3
    bb = b * b
    p = c - bb / 3.0
    q = (2.0 * b * bb) / 27.0 - (b * c) / 3.0 + d

    half_q = 0.5 * q
    delta = half_q * half_q + (p * p * p) / 27.0

    # -----------------------------
    # CASE 1: one real root (Δ >= 0)
    if delta >= 0.0:
        sqrt_delta = np.sqrt(delta)

        # stable Cardano form
        t = -half_q - np.sign(half_q) * sqrt_delta
        u = np.cbrt(t)

        # safe computation of v
        if abs(u) < 1e-30:
            v = np.cbrt(-half_q + sqrt_delta)
        else:
            v = -p / (3.0 * u)

        y = u + v
        return y - b / 3.0

    # -----------------------------
    # CASE 2: three real roots (Δ < 0)
    return -1
    r = np.sqrt(-p / 3.0)

    # safe arccos argument
    arg = -q / (2.0 * r * r * r)
    if arg < -1.0:
        arg = -1.0
    elif arg > 1.0:
        arg = 1.0

    phi = np.arccos(arg)

    # three real roots
    x1 = 2.0 * r * np.cos(phi / 3.0) - b / 3.0
    x2 = 2.0 * r * np.cos((phi + 2.0 * np.pi) / 3.0) - b / 3.0
    x3 = 2.0 * r * np.cos((phi + 4.0 * np.pi) / 3.0) - b / 3.0

    # pick closest to zero
    ax1 = abs(x1)
    ax2 = abs(x2)
    ax3 = abs(x3)

    if ax1 < ax2:
        return x1 if ax1 < ax3 else x3
    else:
        return x2 if ax2 < ax3 else x3

In [13]:
for ii in range(1000):
    sigma_zz = -100000 + np.random.rand() * 100000
    a = 10 * np.exp(np.random.rand() * 3) + 1
    b = a * np.random.rand() * 2
    A = A_fun(-30 + 273.15)
    e1 = np.random.randn() * 1e-4
    e2 = np.random.randn() * 1e-4

    correct = leastsquares_ezz(sigma_zz, a, b, A, e1, e2)

    # ----------------------------------------
    r = r_fun(a, b)
    p = (e1 + e2) / (3 * a) - 3 * (e1 + e2) / (2 * b)
    Asig3 = A * sigma_zz**3
    k0 = Asig3 * ((e1**2 + e2**2 - e1 * e2) / (3 * a) + 3 * (e1 + e2) ** 2 / (4 * b)) + p**3
    k1 = -p * Asig3 - 3 * p**2 * r
    k2 = 0.5 * r * Asig3 + 3 * p * r**2
    k3 = -(r**3)

    myroot = cubic_root_ultrafast(k3, k2, k1, k0)
    # myroot = gagliardini_ezz(sigma_zz, a, b, A, e1, e2)

    if np.abs(1 - myroot / correct) > 0.0001:
        print("-" * 30)
        print("correct lsq", correct)
        print("myroot", myroot)
        print("np.roots", np.roots([k3, k2, k1, k0]))
        print(cubic_root_closest_real_to_zero(k3, k2, k1, k0) / correct)
        print("ERROR for: ", sigma_zz, a, b, A, e1, e2)

# -----------------